In [1]:
from dotenv import load_dotenv
import os
from langchain_groq import ChatGroq
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv()
GROQ_API_KEY=os.environ['GROQ_TOKEN']
GEMINI_API_KEY=os.environ['GEMINI_TOKEN']

In [5]:
response = groq.invoke("berapa suhu di jakarta sekarang?")
response.__dict__

{'content': 'Maaf, saya tidak memiliki akses ke data real‑time. Untuk mengetahui suhu saat ini di Jakarta, Anda bisa cek aplikasi cuaca (misalnya Weather.com, AccuWeather, atau aplikasi cuaca di ponsel Anda) atau situs meteorologi resmi seperti BMKG.',
 'additional_kwargs': {'reasoning_content': 'The user asks in Indonesian: "berapa suhu di Jakarta sekarang?" which means "what is the current temperature in Jakarta?" They want real-time data. As an AI language model, I don\'t have real-time data. According to policy, I cannot provide real-time data. I should respond with a disclaimer: I don\'t have real-time data, but suggest checking a weather website or app. I should not hallucinate a number. So respond politely, mention I can\'t provide real-time info.'},
 'response_metadata': {'token_usage': {'completion_tokens': 163,
   'prompt_tokens': 77,
   'total_tokens': 240,
   'completion_time': 0.165801697,
   'completion_tokens_details': {'reasoning_tokens': 98},
   'prompt_time': 0.003656

In [6]:
response.response_metadata

{'token_usage': {'completion_tokens': 163,
  'prompt_tokens': 77,
  'total_tokens': 240,
  'completion_time': 0.165801697,
  'completion_tokens_details': {'reasoning_tokens': 98},
  'prompt_time': 0.003656782,
  'prompt_tokens_details': None,
  'queue_time': 0.313043039,
  'total_time': 0.169458479},
 'model_name': 'openai/gpt-oss-20b',
 'system_fingerprint': 'fp_228717f27c',
 'service_tier': 'on_demand',
 'finish_reason': 'stop',
 'logprobs': None,
 'model_provider': 'groq'}

In [10]:
from pydantic import BaseModel, Field
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import interrupt, Command, RetryPolicy
from typing import TypedDict, Optional, Literal
import httpx

groq = ChatGroq(
    api_key=GROQ_API_KEY,
    model = "openai/gpt-oss-20b",
    temperature=0.7
)

gemini = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",
    api_key=GEMINI_API_KEY,
    temperature=0.2
)
MAX_VALIDATION_RETRIES = 2


class QueryIntent(BaseModel):
    phenomenon: str = Field(description="main topics studied")
    object_subject: str = Field(description="Object or subject of research, e.g. Twitter, students")
    method: str = Field(description="Implicit methods, e.g. automatic detection, surveys") 
    domain: str = Field(description="Fields of study, e.g. NLP, communications, public health")

class QueryOutput(BaseModel):
    keyword_id: list[str] = Field(description="3-5 formal Indonesian keywords")
    keyword_en: list[str] = Field(description="3-5 formal english keywords")
    boolean_queries: list[str] = Field(..., description="2-3 boolean query variants (ensemble)")

# Agent state (memory)
class ResearchAgentState(TypedDict):
    raw_input: str = Field(description="query input from user")
    intent: Optional[QueryIntent]
    output_query: Optional[QueryOutput]
    validation_hits: Optional[int]
    validation_attempts: int

def extract_intent(state: ResearchAgentState) -> dict:
    structured_llm = groq.with_structured_output(QueryIntent)
    prompt = f""" Analyze the following casual research idea (usually in Indonesian):
            "{state['raw_input']}"

            Identify the main phenomena, objects/subjects, implied methods, and related fields of science.
    """

    intent = structured_llm.invoke(prompt)
    return {"intent": intent}

def generate_queries(state= ResearchAgentState) -> dict:
    structured_llm = groq.with_structured_output(QueryOutput)
    intent = state["intent"]
    prompt = f"""
    Based on the following components:
    Phenomenon: {intent.phenomenon}
    Object/subject: {intent.object_subject}
    Method: {intent.method}
    Field: {intent.domain}

    Create :
    3 to 5 formal academic keywords in Indonesian,
    3 to 5 formal academic keywords in English (standard terms in the field),
    2 to 3 different Boolean query variations (use different synonyms for each variation) for Scopus/Google Scholar

    """

    output = structured_llm.invoke(prompt)
    return {"output_query": output, "validation_attempts": 0}

def human_review_query(state: ResearchAgentState) -> Command[Literal["validate_query", "__end__"]]:
    query: QueryOutput = state.get("output_query")
    query_dict = query.model_dump() if query else {}

    human_decision = interrupt({
        "your_input": state.get("raw_input", ""),
        "keyword_id": query_dict.get("keyword_id", []),
        "keyword_en": query_dict.get("keyword_en", []),
        "boolean_queries": query_dict.get("boolean_queries", []),
        "action": "Please review and approve/edit this query output"
    })

    if human_decision.get("approved"):
        edited = human_decision.get("edited_query")
        updated_output = QueryOutput(**edited) if edited else query
        return Command(
            update={"output_query": updated_output},
            goto="validate_query"
        )
    else:
        return Command(update={}, goto=END)
    

def validate_query(state: ResearchAgentState) -> Command[Literal["human_review_query", "__end__"]]:
    query = state["output_query"].boolean_queries[0]
    attempts = state.get("validation_attempts", 0) + 1

    try:
        resp = httpx.get(
            "https://api.semanticscholar.org/graph/v1/paper/search",
            params={"query": query, "limit": 1},
            timeout=10
        )
        total = resp.json().get("total", 0)
    except Exception:
        total = None

    # kalau hasil kosong/gagal dan belum melebihi batas retry, minta review ulang
    if (total is None or total == 0) and attempts < MAX_VALIDATION_RETRIES:
        return Command(
            update={"validation_hits": total, "validation_attempts": attempts},
            goto="human_review_query"
        )
    else:
        return Command(
            update={"validation_hits": total, "validation_attempts": attempts},
            goto=END  # ganti ke node lanjutan, mis. "fetch_papers", kalau sudah ada
        )

# --- Build graph ---
graph = StateGraph(ResearchAgentState)
graph.add_node("extract_intent", extract_intent)
graph.add_node("generate_queries", generate_queries)
graph.add_node("human_review_query", human_review_query)
graph.add_node(
    "validate_query",
    validate_query,
    retry_policy=RetryPolicy(max_attempts=3)  # untuk kegagalan jaringan sesaat
)

graph.set_entry_point("extract_intent")
graph.add_edge("extract_intent", "generate_queries")
graph.add_edge("generate_queries", "human_review_query")
# human_review_query dan validate_query routingnya dinamis lewat Command,
# jadi tidak perlu add_edge statis lagi untuk keduanya

checkpointer = InMemorySaver()
app = graph.compile(checkpointer=checkpointer)

In [11]:
result = app.invoke({
    "raw_input": "Saya ingin meneliti tentang bagaimana cara mendeteksi hoaks di X"},
    config={"configurable": {
        "thread_id":"session-1"
    }},
    )

print(result)
# print(f"Estimasi jumlah paper relevan: {result['validate_hits']}")

{'raw_input': 'Saya ingin meneliti tentang bagaimana cara mendeteksi hoaks di X', 'intent': QueryIntent(phenomenon='Hoax detection', object_subject='X platform (e.g., Twitter) posts and users', method='Automatic detection, machine learning, natural language processing, content analysis', domain='Social media and misinformation'), 'output_query': QueryOutput(keyword_id=['Deteksi Hoaks', 'Misinformasi Sosial Media', 'Penyelidikan Berbasis Pembelajaran Mesin', 'Analisis Konten Berbasis NLP', 'Platform X'], keyword_en=['Hoax Detection', 'Social Media Misinformation', 'Machine Learning-based Detection', 'Natural Language Processing', 'X Platform'], boolean_queries=['("hoax detection" OR "hoax identification") AND ("X platform" OR "Twitter") AND ("machine learning" OR "artificial intelligence") AND ("natural language processing" OR "NLP") AND ("content analysis" OR "text analysis") AND ("social media" OR "online social network")', '("misinformation detection" OR "fake news detection") AND ("

In [9]:
static = {'raw_input': 'Saya ingin meneliti tentang bagaimana cara mendeteksi hoaks di X', 
          'intent': QueryIntent(
              phenomenon='hoax detection', 
              object_subject='X (Twitter)', 
              method='automatic detection', 
              domain='NLP'), 
            'output_query': QueryOutput(
                keyword_id=['deteksi hoaks', 'Twitter', 'pembelajaran mesin', 'pemrosesan bahasa alami', 'pencatatan otomatis'], 
                keyword_en=['hoax detection', 'Twitter', 'machine learning', 'natural language processing', 'automatic detection'], 
                boolean_queries=['("hoax detection" OR "hoax identification" OR "hoax spotting") AND (Twitter) AND ("automatic detection" OR "automated detection") AND ("natural language processing" OR NLP) AND ("machine learning" OR ML)', '("hoax detection" OR "hoax spotting") AND (Twitter) AND ("automatic detection" OR "automated detection") AND ("natural language processing" OR NLP) AND ("machine learning" OR ML)', '("hoax identification" OR "hoax detection") AND (Twitter) AND ("automatic detection" OR "automated detection") AND ("natural language processing" OR NLP) AND ("machine learning" OR ML)'])}

QueryOutput(keyword_id=['deteksi hoaks', 'Twitter', 'pembelajaran mesin', 'pemrosesan bahasa alami', 'pencatatan otomatis'], keyword_en=['hoax detection', 'Twitter', 'machine learning', 'natural language processing', 'automatic detection'], boolean_queries=['("hoax detection" OR "hoax identification" OR "hoax spotting") AND (Twitter) AND ("automatic detection" OR "automated detection") AND ("natural language processing" OR NLP) AND ("machine learning" OR ML)', '("hoax detection" OR "hoax spotting") AND (Twitter) AND ("automatic detection" OR "automated detection") AND ("natural language processing" OR NLP) AND ("machine learning" OR ML)', '("hoax identification" OR "hoax detection") AND (Twitter) AND ("automatic detection" OR "automated detection") AND ("natural language processing" OR NLP) AND ("machine learning" OR ML)'])